# 00 — Manifest QC + Validation Splits

Build the canonical manifests + held-out validation splits cho AIC 2026 Track 4.

Outputs:

- `{OUTPUT_ROOT}/manifests/train_manifest.parquet` — full 1,013,606 rows
- `{OUTPUT_ROOT}/manifests/train_manifest_trainable.parquet` — 95% (sau val split, dùng để train UIT + LHP)
- `{OUTPUT_ROOT}/manifests/gallery_manifest.parquet` — 36,773
- `{OUTPUT_ROOT}/manifests/query_manifest.parquet` — 1,978
- `{OUTPUT_ROOT}/manifests/val_split.parquet` — ~5% stratified by label_type+scene (in-distribution val)
- `{OUTPUT_ROOT}/manifests/val_zeroshot_scene.parquet` — 1 random scene class hold-out (OOD val)
- `{OUTPUT_ROOT}/manifests/manifest_summary.json`

Deterministic seed=20260514. Val là tín hiệu *tương đối* (Sim2Real gap với test real-world); dùng so sánh configs, không phải tuyệt đối.


In [ ]:
from pathlib import Path
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RUNNING_ON_KAGGLE = Path("/kaggle/input").exists()
if RUNNING_ON_KAGGLE:
    INPUT_ROOT = Path("/kaggle/input")
else:
    local_candidates = [
        Path.cwd() / "Dataset_Structure",
        Path.cwd().parent / "Dataset_Structure",
    ]
    INPUT_ROOT = next((p.resolve() for p in local_candidates if p.exists()), local_candidates[0].resolve())
OUTPUT_ROOT = Path("/kaggle/working/aic2026_track4") if RUNNING_ON_KAGGLE else (Path.cwd() / "aic2026_track4_work").resolve()
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_COUNTS = {
    "train_annotation_files": 75,
    "train_rows": 1_013_606,
    "query_rows": 1_978,
    "query_index_rows": 1_978,
    "gallery_images": 36_773,
}

STRICT_COUNTS = True
REQUIRE_TRAIN_IMAGES = RUNNING_ON_KAGGLE

print("RUNNING_ON_KAGGLE:", RUNNING_ON_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
ACTION_DIRS = {"goal", "full", "wentwrong"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SHARD_RE = re.compile(r"imgs_(\d+)$")


def natural_key(path):
    text = str(path)
    return [int(tok) if tok.isdigit() else tok for tok in re.split(r"(\d+)", text)]


def iter_dirs_limited(root, max_depth=4):
    """Yield directories without descending into large image leaf folders."""
    stack = [(Path(root), 0)]
    while stack:
        path, depth = stack.pop()
        if not path.is_dir():
            continue
        yield path
        if depth >= max_depth:
            continue
        if path.name in ACTION_DIRS or SHARD_RE.match(path.name):
            continue
        try:
            children = [child for child in path.iterdir() if child.is_dir()]
        except PermissionError:
            continue
        stack.extend((child, depth + 1) for child in children)


def find_annotation_train_dir(root):
    candidates = []
    for base in iter_dirs_limited(root, max_depth=6):
        direct = base / "annotation" / "train"
        if direct.exists():
            candidates.append(direct)
        if base.name == "train" and base.parent.name == "annotation":
            candidates.append(base)
    if not candidates:
        raise FileNotFoundError("Could not find annotation/train under INPUT_ROOT")
    return sorted(candidates, key=lambda p: len(str(p)))[0]


def find_test_dir(root):
    candidates = []
    for base in iter_dirs_limited(root, max_depth=6):
        direct = base / "name-masked_test-set"
        if direct.exists():
            candidates.append(direct)
        if base.name == "name-masked_test-set":
            candidates.append(base)
    if not candidates:
        raise FileNotFoundError("Could not find name-masked_test-set under INPUT_ROOT")
    return sorted(candidates, key=lambda p: len(str(p)))[0]


def find_gallery_dir(test_dir):
    candidates = []
    for p in iter_dirs_limited(test_dir, max_depth=4):
        if p.name != "gallery":
            continue
        if p.is_dir() and any(child.suffix.lower() in IMAGE_EXTS for child in p.iterdir() if child.is_file()):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not find gallery image directory")
    return sorted(candidates, key=lambda p: len(str(p)), reverse=True)[0]


annotation_train_dir = find_annotation_train_dir(INPUT_ROOT)
test_dir = find_test_dir(INPUT_ROOT)
gallery_dir = find_gallery_dir(test_dir)
query_text_path = test_dir / "query_text.json"
query_index_path = test_dir / "query_index.txt"

print("annotation_train_dir:", annotation_train_dir)
print("test_dir:", test_dir)
print("gallery_dir:", gallery_dir)
print("query_text_path:", query_text_path)
print("query_index_path:", query_index_path)

In [ ]:
def parse_annotation_image(image_rel):
    parts = image_rel.replace("\\", "/").split("/")
    if len(parts) < 4 or parts[0] != "train":
        raise ValueError(f"Unexpected annotation image path: {image_rel}")
    shard = parts[1]
    action_dir = parts[2]
    image_stem = Path(parts[3]).stem
    return shard, action_dir, image_stem


def build_train_image_index(root):
    index = {}
    duplicate_keys = 0
    shard_dirs = {}

    for base in iter_dirs_limited(root, max_depth=6):
        for parent in (base, base / "train"):
            if not parent.exists():
                continue
            try:
                candidates = [p for p in parent.glob("imgs_*") if p.is_dir() and SHARD_RE.match(p.name)]
            except PermissionError:
                continue
            for shard_dir in candidates:
                if any((shard_dir / action_dir).is_dir() for action_dir in ACTION_DIRS):
                    shard_dirs[str(shard_dir)] = shard_dir

    for shard_dir in tqdm(sorted(shard_dirs.values(), key=natural_key), desc="Index train image shards"):
        shard = shard_dir.name
        for action_dir in ACTION_DIRS:
            image_dir = shard_dir / action_dir
            if not image_dir.is_dir():
                continue
            try:
                files = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
            except PermissionError:
                continue
            for path in files:
                key = (shard, action_dir, path.stem)
                if key in index:
                    duplicate_keys += 1
                    if len(str(path)) < len(str(index[key])):
                        index[key] = path
                else:
                    index[key] = path

    print(f"Indexed train image keys: {len(index):,}; duplicate keys: {duplicate_keys:,}")
    return index


train_image_index = build_train_image_index(INPUT_ROOT)
if REQUIRE_TRAIN_IMAGES and not train_image_index:
    raise RuntimeError("No train images indexed. Add the Part 1-10 train WebP Kaggle datasets as notebook inputs.")

In [ ]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                yield line_no, json.loads(line)


train_files = sorted(annotation_train_dir.glob("imgs_*.json"), key=natural_key)
rows = []

for ann_path in tqdm(train_files, desc="Read train JSONL"):
    for line_no, obj in read_jsonl(ann_path):
        shard, action_dir, image_stem = parse_annotation_image(obj["image"])
        image_path = train_image_index.get((shard, action_dir, image_stem))
        label_type = "anomaly" if "anomaly" in obj else "normal"
        action = obj.get(label_type, "")
        rows.append(
            {
                "row_id": len(rows),
                "image_id": obj.get("image_id", ""),
                "annotation_file": ann_path.name,
                "annotation_line": line_no,
                "annotation_image": obj["image"],
                "shard": shard,
                "action_dir": action_dir,
                "image_stem": image_stem,
                "image_path": str(image_path) if image_path else "",
                "caption": obj.get("caption", ""),
                "scene": obj.get("scene", ""),
                "label_type": label_type,
                "action": action,
                "missing_image": image_path is None,
            }
        )

train_df = pd.DataFrame(rows)
print(train_df.head())
print(train_df.shape)
print(train_df["label_type"].value_counts(dropna=False))
print("missing train image paths:", int(train_df["missing_image"].sum()))

In [ ]:
gallery_files = sorted(
    [p for p in gallery_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS],
    key=natural_key,
)
gallery_df = pd.DataFrame(
    {
        "gallery_order": np.arange(len(gallery_files), dtype=np.int32),
        "gallery_id": [p.stem for p in gallery_files],
        "filename": [p.name for p in gallery_files],
        "image_path": [str(p) for p in gallery_files],
    }
)

query_rows = []
for line_no, obj in read_jsonl(query_text_path):
    query_rows.append(
        {
            "query_order": line_no - 1,
            "query_index": obj["query_index"],
            "caption": obj["caption"],
            "change": obj.get("change", ""),
        }
    )
query_df = pd.DataFrame(query_rows)

with open(query_index_path, "r", encoding="utf-8") as f:
    query_index_list = [line.strip() for line in f if line.strip()]

query_df["query_index_txt"] = query_index_list
query_df["index_matches_json"] = query_df["query_index"] == query_df["query_index_txt"]

print(gallery_df.head())
print(query_df.head())
print("gallery:", gallery_df.shape, "query:", query_df.shape)
print("query order matches:", bool(query_df["index_matches_json"].all()))

In [ ]:
summary = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "running_on_kaggle": RUNNING_ON_KAGGLE,
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "annotation_train_dir": str(annotation_train_dir),
    "test_dir": str(test_dir),
    "gallery_dir": str(gallery_dir),
    "train_annotation_files": len(train_files),
    "train_rows": int(len(train_df)),
    "train_missing_images": int(train_df["missing_image"].sum()),
    "gallery_images": int(len(gallery_df)),
    "query_rows": int(len(query_df)),
    "query_index_rows": int(len(query_index_list)),
    "query_index_matches_json": bool(query_df["index_matches_json"].all()),
}

if STRICT_COUNTS:
    assert summary["train_annotation_files"] == OFFICIAL_COUNTS["train_annotation_files"], summary
    assert summary["train_rows"] == OFFICIAL_COUNTS["train_rows"], summary
    assert summary["gallery_images"] == OFFICIAL_COUNTS["gallery_images"], summary
    assert summary["query_rows"] == OFFICIAL_COUNTS["query_rows"], summary
    assert summary["query_index_rows"] == OFFICIAL_COUNTS["query_index_rows"], summary
    assert summary["query_index_matches_json"], "query_text.json order does not match query_index.txt"

if REQUIRE_TRAIN_IMAGES:
    assert summary["train_missing_images"] == 0, "Some train annotation paths were not mapped to image files"

train_path = MANIFEST_DIR / "train_manifest.parquet"
gallery_path = MANIFEST_DIR / "gallery_manifest.parquet"
query_path = MANIFEST_DIR / "query_manifest.parquet"


def write_parquet(df, path):
    try:
        df.to_parquet(path, index=False)
    except Exception:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
        df.to_parquet(path, index=False)


write_parquet(train_df, train_path)
write_parquet(gallery_df, gallery_path)
write_parquet(query_df, query_path)

with open(MANIFEST_DIR / "manifest_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=True)

print(json.dumps(summary, indent=2))
print("Wrote:", train_path, gallery_path, query_path)

In [ ]:
# === Validation splits ===
#
# 1. val_split: 5% in-distribution, stratified theo (label_type × scene).
# 2. val_zeroshot_scene: 1 random scene class hold-out (proxy cho OOD test gallery).
# 3. train_manifest_trainable: 90% còn lại (loại bỏ cả 2 val splits).
#
# Tất cả ID deterministic qua seed=20260514. Re-run cùng seed → cùng output.
SEED = 20260514
VAL_FRAC = 0.05

rng = np.random.default_rng(SEED)

# Chỉ stratify trên rows có image_path resolved (missing_image=False).
_resolved = train_df[~train_df["missing_image"]].copy().reset_index(drop=True)
print(f"resolved rows for split: {len(_resolved):,}")

# ----- 1. val_zeroshot_scene: random 1 scene class -----
_scenes = sorted([s for s in _resolved["scene"].dropna().unique().tolist() if s])
if len(_scenes) >= 4:
    _zs_scene = _scenes[int(rng.integers(0, len(_scenes)))]
    val_zs_df = _resolved[_resolved["scene"] == _zs_scene].copy().reset_index(drop=True)
    _resolved_after_zs = _resolved[_resolved["scene"] != _zs_scene].copy().reset_index(drop=True)
else:
    # Hiếm: nếu scene field rỗng → skip zero-shot scene split (val_zs sẽ trống).
    _zs_scene = None
    val_zs_df = _resolved.iloc[0:0].copy()
    _resolved_after_zs = _resolved.copy()
print(f"val_zeroshot_scene: scene={_zs_scene!r}, rows={len(val_zs_df):,}")

# ----- 2. val_split: 5% stratified theo (label_type × scene) -----
def _stratified_sample(df, frac, rng):
    out_idx = []
    for _, grp in df.groupby(["label_type", "scene"], dropna=False):
        n = max(1, int(round(len(grp) * frac))) if len(grp) >= 2 else 0
        if n == 0:
            continue
        out_idx.extend(rng.choice(grp.index.values, size=n, replace=False).tolist())
    return df.loc[out_idx].copy().reset_index(drop=True)


val_df = _stratified_sample(_resolved_after_zs, VAL_FRAC, rng)
val_ids = set(val_df["row_id"].tolist())
zs_ids = set(val_zs_df["row_id"].tolist())
print(f"val_split (stratified 5%): rows={len(val_df):,}")

# ----- 3. train_manifest_trainable: full minus both val splits -----
train_trainable_df = (
    train_df[~train_df["row_id"].isin(val_ids | zs_ids)]
    .reset_index(drop=True)
)
print(f"train_trainable rows: {len(train_trainable_df):,}")

assert len(val_ids & zs_ids) == 0, "val_split overlaps with val_zeroshot_scene"
assert len(train_trainable_df) + len(val_df) + len(val_zs_df) == len(train_df) - int(train_df["missing_image"].sum())

# Write parquets
val_path = MANIFEST_DIR / "val_split.parquet"
val_zs_path = MANIFEST_DIR / "val_zeroshot_scene.parquet"
trainable_path = MANIFEST_DIR / "train_manifest_trainable.parquet"
write_parquet(val_df, val_path)
write_parquet(val_zs_df, val_zs_path)
write_parquet(train_trainable_df, trainable_path)

# Augment summary
summary["seed"] = SEED
summary["val_frac"] = VAL_FRAC
summary["val_zeroshot_scene"] = _zs_scene
summary["val_rows"] = int(len(val_df))
summary["val_zeroshot_rows"] = int(len(val_zs_df))
summary["train_trainable_rows"] = int(len(train_trainable_df))

with open(MANIFEST_DIR / "manifest_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=True)

print("Wrote:", val_path, val_zs_path, trainable_path)
print("val_split stratification:")
print(val_df.groupby(["label_type"]).size())


In [ ]:
# Smoke sample for quick visual/path checks.
sample_train = train_df.loc[~train_df["missing_image"]].head(5)[["image_id", "image_path", "caption", "label_type", "action"]]
sample_gallery = gallery_df.head(5)
display(sample_train)
display(sample_gallery)